<a href="https://colab.research.google.com/github/Suvroneel/Phynix-Mood-Based-GenAi-Platform/blob/Suvroneel-patch-1/Phynix_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Phynix Emotion Model - Fine-tuning Script
==========================================
Base model : j-hartmann/emotion-english-distilroberta-base
Dataset    : google-research-datasets/go_emotions (mapped to 7 classes)
Task       : 7-class emotion classification
             anger | disgust | fear | joy | neutral | sadness | surprise

Run:
    pip install transformers datasets scikit-learn torch accelerate
    python finetune_phynix_bert.py
"""


In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

## 1. CONFIG


In [ ]:

BASE_MODEL      = "j-hartmann/emotion-english-distilroberta-base"
OUTPUT_DIR      = "./phynix_emotion_model"
EPOCHS          = 4
BATCH_SIZE      = 16
LR              = 2e-5
MAX_LEN         = 128
SEED            = 42

LABEL_NAMES = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]
LABEL2ID    = {l: i for i, l in enumerate(LABEL_NAMES)}
ID2LABEL    = {i: l for i, l in enumerate(LABEL_NAMES)}

torch.manual_seed(SEED)

## 2. GoEmotions -> 7-class mapping
###    GoEmotions has 28 fine-grained labels.
###    We collapse them into Phynix's 7 categories.

In [ ]:
GO_TO_PHYNIX = {
    # anger cluster
    "anger": "anger", "annoyance": "anger", "disapproval": "anger",
    # disgust cluster
    "disgust": "disgust",
    # fear cluster
    "fear": "fear", "nervousness": "fear",
    # joy cluster
    "joy": "joy", "amusement": "joy", "excitement": "joy",
    "gratitude": "joy", "love": "joy", "optimism": "joy",
    "pride": "joy", "relief": "joy", "admiration": "joy",
    # neutral cluster
    "neutral": "neutral", "realization": "neutral", "approval": "neutral",
    # sadness cluster
    "sadness": "sadness", "grief": "sadness", "remorse": "sadness",
    "disappointment": "sadness", "embarrassment": "sadness",
    # surprise cluster
    "surprise": "surprise", "confusion": "surprise", "curiosity": "surprise",
    "desire": "surprise", "caring": "surprise",
}


def map_go_emotions(example, go_label_names):
    """
    GoEmotions stores labels as a list of ints (multi-label).
    We take the first label and map it to our 7-class scheme.
    Examples with no mapping are dropped later.
    """
    raw_ids = example["labels"]
    if not raw_ids:
        example["label"] = -1
        return example

    # take highest-confidence label (first in list)
    raw_name = go_label_names[raw_ids[0]]
    mapped   = GO_TO_PHYNIX.get(raw_name, None)
    example["label"] = LABEL2ID[mapped] if mapped else -1
    return example


##3. LOAD & PREPROCESS DATASET


In [ ]:

print("Loading GoEmotions dataset...")
raw = load_dataset("google-research-datasets/go_emotions", "simplified")

# Grab the 28 GoEmotions label names from dataset features
go_label_names = raw["train"].features["labels"].feature.names

print("Mapping to 7-class Phynix labels...")
raw = raw.map(lambda x: map_go_emotions(x, go_label_names))

# Drop unmapped examples (label == -1)
raw = raw.filter(lambda x: x["label"] != -1)

print(f"Train size : {len(raw['train'])}")
print(f"Val size   : {len(raw['validation'])}")
print(f"Test size  : {len(raw['test'])}")


Loading GoEmotions dataset...
Mapping to 7-class Phynix labels...
Train size : 43410
Val size   : 5426
Test size  : 5427


## 4. TOKENISE

In [ ]:
print(f"\nLoading tokenizer from {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
    )

tokenized = raw.map(tokenize, batched=True)
tokenized = tokenized.remove_columns("labels")      # drop original GoEmotions multi-label column
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])


Loading tokenizer from j-hartmann/emotion-english-distilroberta-base...


Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

## 5. MODEL


In [ ]:

print(f"\nLoading base model: {BASE_MODEL}")
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,   # classifier head is replaced
)



Loading base model: j-hartmann/emotion-english-distilroberta-base


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 6. METRICS

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")
    return {
        "macro_f1"   : round(macro_f1, 4),
        "weighted_f1": round(weighted_f1, 4),
    }



In [ ]:
# ─────────────────────────────────────────────
# 7. TRAINING ARGS
# ─────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    lr_scheduler_type           = "cosine",
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "macro_f1",
    greater_is_better           = True,
    logging_dir                 = "./logs",
    logging_steps               = 50,
    seed                        = SEED,
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 8. TRAINER

In [ ]:



trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = tokenized["train"],
    eval_dataset    = tokenized["validation"],
    processing_class = tokenizer,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)

print("\nStarting fine-tuning...\n")
trainer.train()


Starting fine-tuning...



Epoch,Training Loss,Validation Loss,Macro F1,Weighted F1
1,0.827821,0.809876,0.630000,0.695400
2,0.738578,0.807712,0.633700,0.701700
3,0.628914,0.830716,0.631400,0.697900
4,0.559220,0.861040,0.632500,0.699100


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=10856, training_loss=0.7262367150233613, metrics={'train_runtime': 941.9997, 'train_samples_per_second': 184.331, 'train_steps_per_second': 11.524, 'total_flos': 5750922527631360.0, 'train_loss': 0.7262367150233613, 'epoch': 4.0})


## 9. EVALUATE ON TEST SET


In [ ]:

print("\nEvaluating on test set...")
predictions = trainer.predict(tokenized["test"])
preds  = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

print("\n── Classification Report ──────────────────")
print(classification_report(labels, preds, target_names=LABEL_NAMES))







Evaluating on test set...



── Classification Report ──────────────────
              precision    recall  f1-score   support

       anger       0.54      0.61      0.57       703
     disgust       0.49      0.45      0.47        84
        fear       0.67      0.67      0.67        90
         joy       0.78      0.89      0.83      1548
     neutral       0.76      0.65      0.70      2033
     sadness       0.60      0.54      0.57       317
    surprise       0.57      0.61      0.59       652

    accuracy                           0.70      5427
   macro avg       0.63      0.63      0.63      5427
weighted avg       0.70      0.70      0.70      5427



In [ ]:
# ─────────────────────────────────────────────
# 10. SAVE
# ─────────────────────────────────────────────
print(f"\nSaving fine-tuned model to {OUTPUT_DIR}...")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Done. Load it with:")
print(f'  AutoModelForSequenceClassification.from_pretrained("{OUTPUT_DIR}")')


Saving fine-tuned model to ./phynix_emotion_model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Load it with:
  AutoModelForSequenceClassification.from_pretrained("./phynix_emotion_model")


In [ ]:
# ─────────────────────────────────────────────
# 11. QUICK INFERENCE TEST
# ─────────────────────────────────────────────
print("\n── Quick inference check ──────────────────")
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model     = OUTPUT_DIR,
    tokenizer = OUTPUT_DIR,
    device    = 0 if torch.cuda.is_available() else -1,
)

test_sentences = [
    "I am so happy today, everything feels amazing!",
    "I feel completely hopeless and empty inside.",
    "This is absolutely disgusting behaviour.",
    "I am not sure how I feel right now.",
    "That really scared me, I was trembling.",
]

for sentence in test_sentences:
    result = clf(sentence)[0]
    print(f"  [{result['label']:>10}  {result['score']:.3f}]  {sentence}")


── Quick inference check ──────────────────


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

  [       joy  0.994]  I am so happy today, everything feels amazing!
  [   sadness  0.937]  I feel completely hopeless and empty inside.
  [   disgust  0.985]  This is absolutely disgusting behaviour.
  [  surprise  0.924]  I am not sure how I feel right now.
  [      fear  0.989]  That really scared me, I was trembling.


In [ ]:
import os
os.listdir("/content/phynix_emotion_model")

['training_args.bin',
 'tokenizer_config.json',
 'config.json',
 'checkpoint-5428',
 'checkpoint-8142',
 'checkpoint-10856',
 'checkpoint-2714',
 'model.safetensors',
 'tokenizer.json']

##Saving

In [ ]:
import shutil

shutil.make_archive("phynix_model", 'zip', "/content/phynix_emotion_model")

'/content/phynix_model.zip'

In [ ]:
import os
os.listdir("/content")

['.config', 'phynix_emotion_model', 'phynix_model.zip', 'sample_data']

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="Suvroneel/phynix-emotion-model"
)

print(classifier("I feel amazing today!"))
print(classifier("I am really sad and tired"))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[{'label': 'joy', 'score': 0.9916365742683411}]
[{'label': 'sadness', 'score': 0.9792345762252808}]
